# nb01 — Exploração da API do parceiro

**O que este notebook faz:** lê a API industrial do parceiro e os YAMLs do corpus, e produz
as três coisas que o resto do framework consome:

1. o **catálogo de respostas por endpoint** — o que vem em cada `mode`, quais campos somem e
   quais não (fonte de `docs/catalogo_respostas.md`);
2. a **varredura de seeds** — para cada um dos 24 cenários, quais das 1000 seeds satisfazem
   todos os `modos_exigidos` (fonte da mesma tabela no catálogo e de
   `docs/reconciliacao_pendente.md`);
3. a figura `figures/fig01_distribuicao_status.png`.

**O notebook não executa o agente.** Só faz `GET`/`POST` na API e lê `scenarios/*.yaml`.

**Pré-requisito: a API tem de estar no ar em `localhost:8000`.** Sem ela nenhuma célula a partir
da §1 roda — não há fixture gravada, de propósito: o valor deste notebook é justamente bater no
servidor de verdade em vez de confiar na réplica de `resolve_mode` que vive em
`scripts/validar_cenarios.py`.

```
make api    # em outro terminal
```

Custo da execução completa: ~70 s, quase tudo na varredura da §5 (67 000 chamadas HTTP locais).

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import time
import urllib.parse
from collections import Counter
from pathlib import Path

import httpx
import pandas as pd
import plotly.graph_objects as go
import yaml

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys

if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from tapieval import figuras as fg

FIGURAS = RAIZ / "figures"
BASE = "http://localhost:8000"
CONTRATO = RAIZ / "inteli-tractian-project" / "agent-input" / "api-contract.openapi.yaml"
SEED_CFG = json.loads((RAIZ / "inteli-tractian-project" / "data" / "seed.json").read_text())
OVERRIDES, DISTRIBUICAO = SEED_CFG["overrides"], SEED_CFG["distribution"]
MODOS = ["complete", "partial", "inconclusive", "conflict", "unavailable"]
SEEDS = [f"s{i:03d}" for i in range(1000)]

pd.set_option("display.max_colwidth", 78)
pd.set_option("display.width", 160)

cli = httpx.Client(base_url=BASE, headers={"x-user-id": "usr_ana"}, timeout=60)
print("API viva:", cli.get("/openapi.json").status_code == 200, "·", RAIZ.name)

API viva: True · case_TRACTIAN


In [2]:
def resolve_mode(recurso: str, categoria: str, seed: str | None) -> str:
    '''Réplica de api/app/prob.py::resolve_mode. A §5 confere se ela bate com a API.'''
    ov = OVERRIDES.get(recurso, {})
    if categoria in ov:
        return ov[categoria]
    if seed == "complete":
        return "complete"
    if seed == "degraded":
        return "partial"
    h = hashlib.sha256(f"{seed or 'noseed'}|{recurso}|{categoria}".encode()).hexdigest()
    r = int(h[:12], 16) / float(0xFFFFFFFFFFFF)
    acumulado = 0.0
    for nome, peso in DISTRIBUICAO.items():
        acumulado += peso
        if r < acumulado:
            return nome
    return "complete"


ROTA_POR_CATEGORIA = {
    "asset": "/assets/{r}",
    "baseline": "/assets/{r}/baseline",
    "rms": "/assets/{r}/rms",
    "spectrum": "/assets/{r}/spectrum",
    "data_quality": "/assets/{r}/data-quality",
    "analyses": "/assets/{r}/analyses",
    "model": "/models/{r}",
    "company": "/companies/{r}",
    "assets": "/companies/{r}/assets",
}


def rota(recurso: str, categoria: str) -> str:
    '''Mesma convenção de scripts/checar_seeds_na_api.py — `recurso` é o path param.'''
    if categoria == "knowledge":
        termo = recurso.split(":", 1)[1]
        if termo.startswith("kb_"):
            return f"/knowledge/{termo}"
        return "/knowledge/search?q=" + urllib.parse.quote(termo)
    return ROTA_POR_CATEGORIA[categoria].format(r=recurso)


def buscar(caminho: str, seed: str | None = None, user: str = "usr_ana") -> dict:
    url = caminho if seed is None else caminho + ("&" if "?" in caminho else "?") + f"seed={seed}"
    return cli.get(url, headers={"x-user-id": user}).json()


def primeira_seed(recurso: str, categoria: str, modo: str) -> str | None:
    return next((s for s in SEEDS if resolve_mode(recurso, categoria, s) == modo), None)


CENARIOS = [
    yaml.safe_load(p.read_text())
    for p in sorted((RAIZ / "scenarios").glob("*.yaml"))
    if not p.name.startswith("_")
]
print(len(CENARIOS), "cenários carregados")

24 cenários carregados


---
## 1. O contrato OpenAPI perde um endpoint no parse (X10)

A chave `/assets/{assetId}` aparece **duas vezes** no `api-contract.openapi.yaml`: um bloco `get`
(`getAsset`) e um bloco `patch` (`updateAssetConfig`). YAML não proíbe chave duplicada em mapa —
`yaml.safe_load` simplesmente deixa a última vencer. O `get` some.

Isto importa porque `get_asset` é a tool mais usada do corpus. Quem gerar o catálogo de tools do
servidor MCP (T13) parseando o YAML com `safe_load` entrega ao agente um catálogo sem ela.

In [3]:
class LoaderTolerante(yaml.SafeLoader):
    '''SafeLoader que **funde** chaves duplicadas em vez de sobrescrever.

    É o contorno mínimo para X10: preserva o `get` e o `patch` de `/assets/{assetId}`.
    '''


def _funde(loader, node, deep=False):
    mapa = {}
    for chave_no, valor_no in node.value:
        chave = loader.construct_object(chave_no, deep=deep)
        valor = loader.construct_object(valor_no, deep=True)
        if chave in mapa and isinstance(mapa[chave], dict) and isinstance(valor, dict):
            mapa[chave] = {**mapa[chave], **valor}
        else:
            mapa[chave] = valor
    return mapa


LoaderTolerante.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, _funde)

texto = CONTRATO.read_text()
ingenuo = yaml.safe_load(texto)["paths"]
tolerante = yaml.load(texto, Loader=LoaderTolerante)["paths"]
por_regex = re.findall(r"operationId:\s*(\w+)", texto)
vivo = cli.get("/openapi.json").json()["paths"]


def ops(paths: dict) -> set[str]:
    return {v["operationId"] for m in paths.values() for v in m.values() if "operationId" in v}


comparacao = pd.DataFrame(
    [
        {"fonte": "yaml.safe_load (ingênuo)", "paths": len(ingenuo), "operations": len(ops(ingenuo)),
         "tem getAsset": "getAsset" in ops(ingenuo)},
        {"fonte": "LoaderTolerante", "paths": len(tolerante), "operations": len(ops(tolerante)),
         "tem getAsset": "getAsset" in ops(tolerante)},
        {"fonte": "regex sobre o texto cru", "paths": None, "operations": len(por_regex),
         "tem getAsset": "getAsset" in por_regex},
        {"fonte": "/openapi.json da API viva", "paths": len(vivo), "operations": len(ops(vivo)),
         "tem getAsset": "getAsset" in ops(vivo)},
    ]
)
display(comparacao)
print("\nperdido pelo safe_load:", sorted(set(por_regex) - ops(ingenuo)))
print("métodos de /assets/{assetId} — safe_load:", sorted(ingenuo["/assets/{assetId}"]),
      "· tolerante:", sorted(tolerante["/assets/{assetId}"]))
print("GET /assets/asset_M101 na API real:", cli.get("/assets/asset_M101?seed=complete").status_code)

,fonte,paths,operations,tem getAsset
0,yaml.safe_load (ingênuo),17.0,17,False
1,LoaderTolerante,17.0,18,True
2,regex sobre o texto cru,NaN,18,True
3,/openapi.json da API viva,17.0,18,False



perdido pelo safe_load: ['getAsset']
métodos de /assets/{assetId} — safe_load: ['patch'] · tolerante: ['get', 'patch']
GET /assets/asset_M101 na API real: 200


**Verificado.** `safe_load` devolve 17 paths e 17 operações; o texto cru tem 18 `operationId`, e a
API real serve os dois métodos (`GET` responde 200). A operação perdida é exatamente `getAsset`.

O `/openapi.json` que a API gera **não** serve de substituto: ele expõe os dois métodos, mas com
`operationId` autogerado pelo FastAPI (`get_asset_assets__asset_id__get`), e a convenção do corpus
é *nome de tool = `operationId` do contrato em snake_case* (`scenarios/README.md`). Trocar de fonte
renomearia todas as 18 tools.

**Contornos aceitáveis, em ordem de preferência:**

1. `LoaderTolerante` acima — funde chaves duplicadas, mantém os `operationId` do contrato;
2. regex `operationId:\s*(\w+)` sobre o texto cru — é o que `scripts/validar_cenarios.py` já faz,
   e por isso o validador do corpus **não** é vítima do X10;
3. corrigir o YAML do parceiro — fora de escopo, é material recebido.

> Uma segunda cópia bit a bit idêntica do contrato vive em
> `inteli-tractian-project/docs/api-contract.openapi.yaml`, e é essa que o
> `validar_cenarios.py` lê. Duas cópias, uma fonte de verdade: T13 precisa escolher uma.

---
## 2. O envelope `{mode, notes, data}`

Toda leitura devolve o envelope. O `mode` chega **explícito** — o classificador de status (T7)
lê o campo, não infere pela forma do corpo. O que sobra a inferir é *quais campos sumiram*, e é
disso que trata esta seção.

Sondamos cada categoria nos cinco modos, escolhendo para cada uma a primeira seed que a produz.

In [4]:
ALVOS = [
    ("company", "comp_forja_br", "get_company"),
    ("assets", "comp_forja_br", "list_assets_by_company"),
    ("asset", "asset_M101", "get_asset"),
    ("analyses", "asset_M101", "list_analyses"),
    ("baseline", "asset_M101", "get_baseline"),
    ("rms", "asset_M101", "get_rms_series"),
    ("spectrum", "asset_M101", "get_spectrum"),
    ("data_quality", "asset_M101", "get_data_quality"),
    ("model", "mdl_vib_v3", "get_model"),
    ("knowledge", "knowledge:BPFO", "search_knowledge"),
    ("knowledge", "knowledge:kb_proc_001", "get_knowledge_doc"),
]

amostras: dict[tuple[str, str], dict] = {}
linhas = []
for categoria, recurso, tool in ALVOS:
    caminho = rota(recurso, categoria)
    for modo in MODOS:
        seed = primeira_seed(recurso, categoria, modo)
        corpo = buscar(caminho, seed)
        amostras[(tool, modo)] = corpo
        d = corpo["data"]
        linhas.append({
            "tool": tool, "categoria": categoria, "modo pedido": modo,
            "mode devolvido": corpo["mode"],
            "chaves de data": ", ".join(sorted(d)) if isinstance(d, dict) and d else "{} (vazio)",
        })

catalogo = pd.DataFrame(linhas)
display(catalogo.style.hide(axis="index"))

tool,categoria,modo pedido,mode devolvido,chaves de data
get_company,company,complete,complete,"id, name, segment, timezone"
get_company,company,partial,partial,"id, name, segment, timezone"
get_company,company,inconclusive,inconclusive,"id, name, segment, timezone"
get_company,company,conflict,conflict,"conflict, id, name, segment, timezone"
get_company,company,unavailable,unavailable,"id, name, segment, timezone"
list_assets_by_company,assets,complete,complete,assets
list_assets_by_company,assets,partial,partial,assets
list_assets_by_company,assets,inconclusive,inconclusive,assets
list_assets_by_company,assets,conflict,conflict,"assets, conflict"
list_assets_by_company,assets,unavailable,unavailable,assets


In [5]:
notas = (
    catalogo.assign(notes=[amostras[(r["tool"], r["modo pedido"])]["notes"]
                           for _, r in catalogo.iterrows()])
    .groupby(["modo pedido", "notes"], dropna=False).size().rename("ocorrências").reset_index()
)
display(notas.style.hide(axis="index"))

modo pedido,notes,ocorrências
complete,nan,11
conflict,Conflito entre fontes: verifique análises especializadas.,11
inconclusive,Resultado inconclusivo: dados insuficientes para concluir.,7
inconclusive,Resultado inconclusivo: verifique fontes complementares.,4
partial,Informação parcial: campos ausentes (detalhes),7
partial,Informação parcial: campos ausentes ['features'],1
partial,Informação parcial: campos ausentes ['freshness_minutes'],1
partial,"Informação parcial: campos ausentes ['last_run_at', 'requirements']",1
partial,Informação parcial: campos ausentes ['samples'],1
unavailable,Indisponibilidade temporária parcial; dados podem estar incompletos.,4


As `notes` são **texto fixo por (modo, estabilidade da categoria)** — não descrevem o corpo. Três
categorias são *estáveis* (`knowledge`, `company`, `assets`, em `main.py::_apply_mode`) e recebem
uma nota mais branda em `inconclusive` e `unavailable`, mantendo o payload inteiro.

In [6]:
def campos(corpo: dict) -> set[str]:
    d = corpo["data"]
    return set(d) if isinstance(d, dict) else set()


diffs = []
for categoria, recurso, tool in ALVOS:
    base = campos(amostras[(tool, "complete")])
    linha = {"tool": tool, "campos em complete": len(base)}
    for modo in MODOS[1:]:
        atual = campos(amostras[(tool, modo)])
        sumiram = sorted(base - atual)
        surgiram = sorted(atual - base)
        marca = ", ".join(f"-{c}" for c in sumiram) + ("; " if sumiram and surgiram else "")
        marca += ", ".join(f"+{c}" for c in surgiram)
        linha[modo] = marca or "(idêntico)"
    diffs.append(linha)

display(pd.DataFrame(diffs).style.hide(axis="index"))

tool,campos em complete,partial,inconclusive,conflict,unavailable
get_company,4,(idêntico),(idêntico),+conflict,(idêntico)
list_assets_by_company,1,(idêntico),(idêntico),+conflict,(idêntico)
get_asset,17,(idêntico),"-bearing_pn, -bpfi_hz, -bpfo_hz, -bsf_hz, -company_id, -criticality, -ftf_hz, -id, -line, -line_frequency_hz, -machine_type, -name, -parent_asset_id, -plant, -points, -rotation_rpm, -sensor_status; +inconclusive",+conflict,"-bearing_pn, -bpfi_hz, -bpfo_hz, -bsf_hz, -company_id, -criticality, -ftf_hz, -id, -line, -line_frequency_hz, -machine_type, -name, -parent_asset_id, -plant, -points, -rotation_rpm, -sensor_status"
list_analyses,1,(idêntico),-analyses; +inconclusive,+conflict,-analyses
get_baseline,10,-features,"-detection_mode, -established_at, -features, -id, -invalidated_at, -invalidation_reason, -learnable, -point_id, -state; +inconclusive",+conflict,"-asset_id, -detection_mode, -established_at, -features, -id, -invalidated_at, -invalidation_reason, -learnable, -point_id, -state"
get_rms_series,7,-samples,"-alarm_threshold, -baseline_reference, -baseline_state, -point_id, -samples, -unit; +inconclusive",+conflict,"-alarm_threshold, -asset_id, -baseline_reference, -baseline_state, -point_id, -samples, -unit"
get_spectrum,5,(idêntico),"-bands_missing, -collected_at, -peaks, -point_id; +inconclusive",+conflict,"-asset_id, -bands_missing, -collected_at, -peaks, -point_id"
get_data_quality,6,-freshness_minutes,"-completeness, -freshness_minutes, -point_id, -snr_db, -staleness_flag; +inconclusive",+conflict,"-asset_id, -completeness, -freshness_minutes, -point_id, -snr_db, -staleness_flag"
get_model,6,"-last_run_at, -requirements","-coverage, -id, -last_run_at, -processing_state, -requirements, -version; +inconclusive",+conflict,"-coverage, -id, -last_run_at, -processing_state, -requirements, -version"
search_knowledge,1,(idêntico),(idêntico),+conflict,(idêntico)


Lendo a tabela, as quatro formas de degradação:

| Modo | Forma do `data` |
|---|---|
| `partial` | payload menos os campos de `_PARTIAL_DROP[categoria]` — **e nada, se a categoria não tem entrada** |
| `inconclusive` | categoria instável: `{"inconclusive": true}` + `asset_id` **se o payload original o tinha**. Categoria estável: payload inteiro |
| `conflict` | payload inteiro **+ `"conflict": true`** — o único marcador; nada é removido |
| `unavailable` | categoria instável: `data == {}`. Categoria estável: payload inteiro |

Três consequências que o classificador (T7) precisa embutir:

- **`conflict` só se detecta pela chave `data.conflict`** (ou pelo `mode`). Nas listagens ela é
  *irmã* da lista, não vai dentro dos itens: `{"analyses": [...], "conflict": true}`.
- **A forma do `inconclusive` não é uma só.** `{"inconclusive": true}` sozinho quando o payload
  não tinha `asset_id` (`asset`, `model`, `analyses`), com `asset_id` quando tinha
  (`baseline`, `rms`, `spectrum`, `data_quality`) — e ainda uma terceira forma, `{"<recurso>":
  null}`, tratada na §4.
- **`unavailable` esvazia `data` sem erro HTTP.** Continua 200.

---
## 3. X5 — `mode=partial` que não tira campo nenhum

A `notes` de `partial` sempre anuncia lacuna. `_PARTIAL_DROP` só tem entrada para cinco
categorias; nas outras a nota mente por construção. Isto é armadilha proposital em CEN-11/12/13:
o agente que "declara a lacuna" nesses casos está **alucinando** uma.

In [7]:
falsos = []
for categoria, recurso, tool in ALVOS:
    base, parc = campos(amostras[(tool, "complete")]), campos(amostras[(tool, "partial")])
    sumiram = sorted(base - parc)
    falsos.append({
        "tool": tool, "categoria": categoria,
        "campos removidos": ", ".join(sumiram) if sumiram else "NENHUM",
        "aviso falso?": "sim" if not sumiram else "não",
        "notes": amostras[(tool, "partial")]["notes"],
    })
display(pd.DataFrame(falsos).style.hide(axis="index"))

tool,categoria,campos removidos,aviso falso?,notes
get_company,company,NENHUM,sim,Informação parcial: campos ausentes (detalhes)
list_assets_by_company,assets,NENHUM,sim,Informação parcial: campos ausentes (detalhes)
get_asset,asset,NENHUM,sim,Informação parcial: campos ausentes (detalhes)
list_analyses,analyses,NENHUM,sim,Informação parcial: campos ausentes (detalhes)
get_baseline,baseline,features,não,Informação parcial: campos ausentes ['features']
get_rms_series,rms,samples,não,Informação parcial: campos ausentes ['samples']
get_spectrum,spectrum,NENHUM,sim,Informação parcial: campos ausentes (detalhes)
get_data_quality,data_quality,freshness_minutes,não,Informação parcial: campos ausentes ['freshness_minutes']
get_model,model,"last_run_at, requirements",não,"Informação parcial: campos ausentes ['last_run_at', 'requirements']"
search_knowledge,knowledge,NENHUM,sim,Informação parcial: campos ausentes (detalhes)


### O caso que a documentação ainda não registrava: `list_analyses`

`_PARTIAL_DROP["analyses"] = ("evidence", "limitations")`, e `CENARIOS §5.4/§7.5` tratam
`analyses` como categoria que **remove** campo. Isso vale para `GET /analyses/{id}`, onde o
payload é a análise. Não vale para `GET /assets/{id}/analyses`, onde o payload é
`{"analyses": [...]}`: o corte é aplicado às chaves de **primeiro nível**, e `evidence`/
`limitations` moram dentro dos itens. A lista passa intacta.

In [8]:
seed_m208 = "s004"  # env_seed canônica de cen_04; asset_M208 tem override analyses=partial
lista = buscar("/assets/asset_M208/analyses", seed_m208)
item_id = lista["data"]["analyses"][0]["id"]
unico = buscar(f"/analyses/{item_id}", seed_m208)

display(pd.DataFrame([
    {"endpoint": "GET /assets/asset_M208/analyses", "tool": "list_analyses", "mode": lista["mode"],
     "evidence/limitations presentes?": all(
         k in lista["data"]["analyses"][0] for k in ("evidence", "limitations")),
     "notes": lista["notes"]},
    {"endpoint": f"GET /analyses/{item_id}", "tool": "get_analysis", "mode": unico["mode"],
     "evidence/limitations presentes?": all(k in unico["data"] for k in ("evidence", "limitations")),
     "notes": unico["notes"]},
]).style.hide(axis="index"))

endpoint,tool,mode,evidence/limitations presentes?,notes
GET /assets/asset_M208/analyses,list_analyses,partial,True,Informação parcial: campos ausentes (detalhes)
GET /analyses/an_9905,get_analysis,partial,False,"Informação parcial: campos ausentes ['evidence', 'limitations']"


**Mesmo recurso, mesmo `mode`, mesma seed — corpos diferentes.** O corte de `partial` é função do
*endpoint*, não da categoria. Para T7: `campos_ausentes` tem de ser calculado contra o schema do
**recurso que o endpoint devolve**, e o mapa correto é este:

| Endpoint | Categoria | `partial` remove |
|---|---|---|
| `GET /analyses/{id}` | `analyses` | `evidence`, `limitations` |
| `GET /assets/{id}/analyses` | `analyses` | **nada** |
| `GET /assets/{id}/baseline` | `baseline` | `features` |
| `GET /assets/{id}/rms` | `rms` | `samples` |
| `GET /assets/{id}/data-quality` | `data_quality` | `freshness_minutes` |
| `GET /models/{id}` | `model` | `requirements`, `last_run_at` |
| todos os demais | `asset`, `spectrum`, `company`, `assets`, `knowledge` | **nada** |

---
## 4. Quatro exceções que quebram quem assume "todo GET tem envelope"

In [9]:
excecoes = []

# (a) /users/me nao tem envelope
bruto = cli.get("/users/me", headers={"x-user-id": "usr_ana"}).json()
excecoes.append({"exceção": "GET /users/me não tem envelope",
                 "evidência": f"chaves = {sorted(bruto)} · 'mode' presente: {'mode' in bruto}"})

# (b) piso inconclusive: linha ausente vence a seed
piso = []
for ativo in ["asset_M101", "asset_M102"]:
    for sufixo, categoria in [("baseline", "baseline"), ("spectrum", "spectrum"),
                              ("data-quality", "data_quality")]:
        r = buscar(f"/assets/{ativo}/{sufixo}", "complete")
        if r["mode"] != "complete":
            piso.append(f"{ativo}/{categoria} → {r['mode']} {r['data']} (seed=complete!)")
excecoes.append({"exceção": "linha ausente força inconclusive, ignorando a seed",
                 "evidência": " · ".join(piso)})

# (c) permissao e checada antes da justificativa
r = cli.post("/analyses/an_9901/reprocess", json={"justification": "curta"},
             headers={"x-user-id": "usr_ana"})
excecoes.append({"exceção": "403 de permissão vem antes do 400 de justificativa",
                 "evidência": f"usr_ana (sem action_low) + justificativa de 5 chars → "
                              f"{r.status_code} {r.json()['code']}"})

# (d) leitura nao exige x-user-id
r = cli.get("/assets/asset_X216?seed=complete", headers={"x-user-id": ""})
excecoes.append({"exceção": "leitura não exige (nem filtra por) x-user-id",
                 "evidência": f"sem header → {r.status_code}, company_id do ativo = "
                              f"{r.json()['data']['company_id']}"})

display(pd.DataFrame(excecoes).style.hide(axis="index"))

exceção,evidência
GET /users/me não tem envelope,"chaves = ['company_id', 'id', 'name', 'permissions', 'role'] · 'mode' presente: False"
"linha ausente força inconclusive, ignorando a seed",asset_M102/spectrum → inconclusive {'spectrum': None} (seed=complete!)
403 de permissão vem antes do 400 de justificativa,usr_ana (sem action_low) + justificativa de 5 chars → 403 FORBIDDEN
leitura não exige (nem filtra por) x-user-id,"sem header → 200, company_id do ativo = comp_cimento_vale"


A segunda é a que morde a réplica de `resolve_mode`: quando `store` não acha a linha, `main.py`
devolve `envelope(..., Mode.INCONCLUSIVE, ...)` **antes** de consultar a seed, com uma terceira
forma de `data` (`{"spectrum": null}`). No dataset atual isso atinge um único par —
`asset_M102`/`spectrum` — e nenhum cenário depende dele. A §5 confirma que é o único.

---
## 5. Varredura: réplica × API real, 1000 seeds

Os 24 cenários exigem modos sobre um conjunto de pares `(recurso, categoria)`. Varremos **todos**
esses pares contra as 1000 seeds na API de verdade, e comparamos com a réplica de `resolve_mode`.
É a célula cara do notebook (~60 s).

In [10]:
pares = sorted({(e["recurso"], e["categoria"])
                for c in CENARIOS for e in c["ambiente"].get("modos_exigidos", [])})

t0 = time.time()
modo_api: dict[tuple[str, str, str], str] = {}
divergencias = []
for recurso, categoria in pares:
    caminho = rota(recurso, categoria)
    sep = "&" if "?" in caminho else "?"
    for seed in SEEDS:
        obtido = cli.get(f"{caminho}{sep}seed={seed}").json()["mode"]
        modo_api[(recurso, categoria, seed)] = obtido
        if obtido != resolve_mode(recurso, categoria, seed):
            divergencias.append((recurso, categoria, seed,
                                 resolve_mode(recurso, categoria, seed), obtido))

print(f"{len(pares)} pares × {len(SEEDS)} seeds = {len(modo_api):,} chamadas em {time.time()-t0:.0f}s")
print(f"divergências réplica × API: {len(divergencias)}")

67 pares × 1000 seeds = 67,000 chamadas em 63s
divergências réplica × API: 0


**Zero divergências.** A réplica de `resolve_mode` em `scripts/validar_cenarios.py` é fiel à API
para todos os pares que o corpus usa. O piso `inconclusive` da §4 não aparece porque nenhum
cenário exige modo de um par cuja linha esteja faltando.

In [11]:
observado = Counter(modo_api.values())
total = sum(observado.values())

com_override = {(r, c) for r, cats in OVERRIDES.items() for c in cats}
por_categoria: dict[str, Counter] = {}
for (recurso, categoria, _), modo in modo_api.items():
    por_categoria.setdefault(categoria, Counter())[modo] += 1

resumo = pd.DataFrame({
    "nominal (seed.json)": pd.Series(DISTRIBUICAO),
    "observado": pd.Series({m: observado[m] / total for m in MODOS}),
}).reindex(MODOS)
resumo["desvio p.p."] = ((resumo["observado"] - resumo["nominal (seed.json)"]) * 100).round(1)
display(resumo.style.format({"nominal (seed.json)": "{:.1%}", "observado": "{:.1%}"}))
print("pares com override fixo entre os varridos:",
      sorted(p for p in pares if p in com_override))

,nominal (seed.json),observado,desvio p.p.
complete,60.0%,52.4%,-7.600000
partial,15.0%,20.3%,5.300000
inconclusive,10.0%,10.2%,0.200000
conflict,8.0%,9.7%,1.700000
unavailable,7.0%,7.4%,0.400000


pares com override fixo entre os varridos: [('asset_C710', 'rms'), ('asset_G501', 'analyses'), ('asset_G501', 'baseline'), ('asset_G501', 'data_quality'), ('asset_G501', 'rms'), ('asset_M205', 'analyses'), ('asset_M208', 'analyses'), ('asset_M605', 'spectrum'), ('asset_S420', 'analyses'), ('asset_V301', 'data_quality')]


In [12]:
# Paleta categórica em ordem fixa (slots 1-5 do sistema de design), validada para pares
# adjacentes em pilha. Contraste sub-3:1 em três slots => rótulos diretos, nunca só cor.
# Os cinco modos de retorno da API. Os três que existem na paleta do trabalho vêm do
# módulo; os dois que só este notebook usa ficam declarados aqui, e é o teste da T31
# que impede um deles de virar um segundo azul.
CORES = {"complete": fg.AZUL, "partial": "#eb6834", "inconclusive": fg.VERDE,
         "conflict": fg.AMBAR, "unavailable": "#e87ba4"}
TINTA, TINTA2, SUPERFICIE = fg.TINTA, fg.TINTA2, fg.SUPERFICIE

ORDEM = ["knowledge", "company", "assets", "asset", "model", "analyses",
         "baseline", "rms", "spectrum", "data_quality"]
ordem = [c for c in ORDEM if c in por_categoria]

rotulos = ["nominal (seed.json)", f"observado ({total//1000}k chamadas)", ""] + [
    f"{c}{' ◆' if any(p[1] == c and p in com_override for p in pares) else ''}" for c in ordem
]
series = {m: [DISTRIBUICAO[m], observado[m] / total, None] +
             [por_categoria[c][m] / sum(por_categoria[c].values()) for c in ordem]
          for m in MODOS}

fig = go.Figure()
for modo in MODOS:
    vals = series[modo]
    fig.add_bar(
        y=rotulos, x=vals, name=modo, orientation="h",
        marker=dict(color=CORES[modo], line=dict(color=SUPERFICIE, width=2)),
        text=[f"{v:.0%}" if v is not None and v >= 0.06 else "" for v in vals],
        textposition="inside", insidetextanchor="middle",
        textfont=dict(color=SUPERFICIE, size=11, family=fg.FAMILIA),
        hovertemplate="%{y} · " + modo + " · %{x:.1%}<extra></extra>",
    )

fig.update_layout(
    barmode="stack", bargap=0.34,
    title=dict(
        text="<b>Distribuição dos modos de retorno da API</b><br>"
             "<span style='font-size:12px;color:#52514e'>67 pares (recurso, categoria) × 1000 "
             "seeds = 67 000 chamadas · ◆ categoria com override fixo em seed.json</span>",
        x=0, xanchor="left", font=dict(size=17, color=TINTA),
    ),
    legend=dict(orientation="h", y=-0.12, x=0, font=dict(color=TINTA2, size=12),
                title=None, traceorder="normal"),
    xaxis=dict(tickformat=".0%", showgrid=True, gridcolor="#e8e7e3", zeroline=False,
               range=[0, 1], tickfont=dict(color=TINTA2, size=11)),
    yaxis=dict(autorange="reversed", tickfont=dict(color=TINTA, size=12)),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
    margin=dict(l=170, r=28, t=86, b=64), height=470, width=fg.LARGURA,
    font=dict(family=fg.FAMILIA),
)

png, svg = fg.exportar(fig, "fig01_distribuicao_status", FIGURAS)
print("gravado:", png.relative_to(RAIZ), "+", svg.name)
fig.show()

gravado: figures/fig01_distribuicao_status.png + fig01_distribuicao_status.svg


A leitura da figura: as três categorias **sem** override seguem a distribuição nominal de perto —
o hash é bem comportado. As categorias com override (`◆`) são puxadas para o modo fixado, porque
`resolve_mode` consulta `overrides` **antes** da seed e retorna direto. É a mecânica que torna
alguns cenários insatisfazíveis por qualquer seed, e é o único caso de "irreparável" previsto em
`scenarios/README.md`.

---
## 6. Varredura de seeds por cenário

Para cada cenário: quantas das 1000 seeds satisfazem **todos** os `modos_exigidos`, e quais. O
resultado alimenta `ambiente.seeds_equivalentes` (bateria de ambiente) e é a base da reconciliação.

In [13]:
tabela = []
for c in CENARIOS:
    amb = c["ambiente"]
    exigidos = amb.get("modos_exigidos", [])
    validas = [s for s in SEEDS
               if all(modo_api[(e["recurso"], e["categoria"], s)] in e["modos"] for e in exigidos)]
    canonica = amb["env_seed"]
    declaradas = amb.get("seeds_equivalentes") or []
    tabela.append({
        "cenário": c["id"], "proc": c["procedencia"], "env_seed": canonica,
        "canônica válida?": canonica in validas,
        "válidas/1000": len(validas), "%": round(len(validas) / 10, 1),
        "equivalentes declaradas": len(declaradas),
        "declaradas inválidas": [s for s in declaradas if s not in validas],
        "_validas": validas,
    })

varredura = pd.DataFrame(tabela)
display(varredura.drop(columns=["_validas"]).style.hide(axis="index"))
print("cenários com env_seed inválida:", int((~varredura["canônica válida?"]).sum()))
print("cenários com alguma seeds_equivalentes inválida:",
      int(varredura["declaradas inválidas"].map(bool).sum()))

cenário,proc,env_seed,canônica válida?,válidas/1000,%,equivalentes declaradas,declaradas inválidas
aut_01_barulho_sem_desvio,autoral,s001,True,71,7.100000,7,[]
aut_02_retreinar_sem_base,autoral,s006,True,83,8.300000,7,[]
aut_03_pergunta_que_parece_ordem,autoral,s002,True,330,33.000000,7,[]
aut_04_ativo_de_outra_empresa,autoral,s002,True,365,36.500000,7,[]
aut_05_ativo_inexistente,autoral,s002,True,597,59.700000,7,[]
aut_06_premissa_falsa,autoral,s004,True,46,4.600000,7,[]
aut_07_solicitacao_ambigua,autoral,s002,True,598,59.800000,7,[]
aut_08_acao_errada_sem_permissao,autoral,s025,True,73,7.300000,7,[]
cen_01_quebra_sem_aviso,oficial,s002,True,364,36.400000,6,[]
cen_02_rms_subindo_sem_insight,oficial,s002,True,110,11.000000,6,[]


cenários com env_seed inválida: 0
cenários com alguma seeds_equivalentes inválida: 0


In [14]:
# Tabela em markdown para docs/catalogo_respostas.md — as 8 primeiras válidas por cenário.
print("| cenário | `env_seed` canônica | válidas/1000 | primeiras 8 seeds válidas |")
print("|---|---|---|---|")
for _, r in varredura.iterrows():
    primeiras = ", ".join(f"`{s}`" for s in r["_validas"][:8])
    print(f"| `{r['cenário']}` | `{r['env_seed']}` | {r['válidas/1000']} ({r['%']}%) | {primeiras} |")

| cenário | `env_seed` canônica | válidas/1000 | primeiras 8 seeds válidas |
|---|---|---|---|
| `aut_01_barulho_sem_desvio` | `s001` | 71 (7.1%) | `s001`, `s013`, `s015`, `s021`, `s048`, `s053`, `s081`, `s091` |
| `aut_02_retreinar_sem_base` | `s006` | 83 (8.3%) | `s006`, `s010`, `s019`, `s022`, `s040`, `s053`, `s084`, `s093` |
| `aut_03_pergunta_que_parece_ordem` | `s002` | 330 (33.0%) | `s002`, `s003`, `s005`, `s020`, `s023`, `s025`, `s026`, `s027` |
| `aut_04_ativo_de_outra_empresa` | `s002` | 365 (36.5%) | `s001`, `s002`, `s004`, `s007`, `s009`, `s013`, `s017`, `s018` |
| `aut_05_ativo_inexistente` | `s002` | 597 (59.7%) | `s002`, `s003`, `s005`, `s010`, `s011`, `s012`, `s013`, `s016` |
| `aut_06_premissa_falsa` | `s004` | 46 (4.6%) | `s004`, `s023`, `s032`, `s055`, `s079`, `s089`, `s116`, `s206` |
| `aut_07_solicitacao_ambigua` | `s002` | 598 (59.8%) | `s000`, `s001`, `s002`, `s003`, `s004`, `s006`, `s008`, `s010` |
| `aut_08_acao_errada_sem_permissao` | `s025` | 73 (7.3%) | `s02

---
## 7. As evidências obrigatórias existem no dado real?

`evidencias_obrigatorias` alimenta N1.3. `validar_cenarios.py` só confere que o campo **existe no
schema**; aqui conferimos que ele chega **preenchido** na `env_seed` canônica.

In [15]:
SEM_CAMPO = {"analyses[]", "assets[]", "knowledge", "knowledge.results[]"}
FONTE = {
    "asset": "/assets/{a}", "baseline": "/assets/{a}/baseline", "rms": "/assets/{a}/rms",
    "data_quality": "/assets/{a}/data-quality", "spectrum": "/assets/{a}/spectrum",
    "model": "/models/mdl_vib_v3",
}

achados = []
for c in CENARIOS:
    seed, ativo, usuario = c["ambiente"]["env_seed"], c.get("asset_id"), c["user_id"]
    args = c["gabarito"].get("args_esperados") or {}
    an_id = (args.get("get_analysis") or {}).get("analysis_id")
    for ev in c["gabarito"].get("evidencias_obrigatorias", []):
        recurso, _, campo = ev.partition(".")
        if ev in SEM_CAMPO:
            if ev == "analyses[]" and ativo:
                r = buscar(f"/assets/{ativo}/analyses", seed)
                itens = r["data"].get("analyses")
                if itens is None:
                    achados.append((c["id"], ev, f"lista ausente (mode={r['mode']})"))
                elif not itens:
                    achados.append((c["id"], ev, f"lista VAZIA (mode={r['mode']})"))
            continue
        if recurso == "user":
            dado, modo = cli.get("/users/me", headers={"x-user-id": usuario}).json(), "n/a"
        elif recurso == "analyses[]":
            r = buscar(f"/analyses/{an_id}", seed) if an_id else buscar(f"/assets/{ativo}/analyses", seed)
            modo = r["mode"]
            dado = r["data"] if an_id else ((r["data"].get("analyses") or [{}])[0])
        else:
            r = buscar(FONTE[recurso].format(a=ativo), seed)
            dado, modo = r["data"], r["mode"]
        alvo, resto = dado, campo
        while "." in resto:
            chave, _, resto = resto.partition(".")
            alvo = alvo.get(chave) if isinstance(alvo, dict) else None
        if not isinstance(alvo, dict) or resto not in alvo:
            achados.append((c["id"], ev, f"AUSENTE (mode={modo})"))
        elif alvo[resto] in (None, [], {}):
            achados.append((c["id"], ev, f"presente mas vazio (mode={modo})"))

display(pd.DataFrame(achados, columns=["cenário", "evidência", "o que a API devolve"])
        .style.hide(axis="index"))
print(f"{len(achados)} evidências não chegam preenchidas — todas do tipo `analyses[]`.")

cenário,evidência,o que a API devolve
aut_06_premissa_falsa,analyses[],lista VAZIA (mode=complete)
aut_08_acao_errada_sem_permissao,analyses[],lista VAZIA (mode=complete)
cen_01_quebra_sem_aviso,analyses[],lista ausente (mode=inconclusive)
cen_10_escalar_para_humano,analyses[],lista ausente (mode=inconclusive)


4 evidências não chegam preenchidas — todas do tipo `analyses[]`.


As quatro ocorrências **não são erro de seed** — são intencionais e estão documentadas
(`CENARIOS §7.7` e `§7.9`, `estado_esperado.analises: []` nos YAMLs). O problema que elas criam é
de *métrica*, não de ambiente: `analyses[]` é uma evidência satisfeita por **olhar**, não por
**achar**. Uma implementação ingênua de N1.3 ("a lista veio não-vazia") reprova o agente correto
em `aut_06`, `aut_08`, `cen_01` e `cen_10`. Levantado em `docs/reconciliacao_pendente.md`.

---
## 8. Saídas

| Saída | Onde |
|---|---|
| catálogo de respostas por endpoint | `docs/catalogo_respostas.md` (§2–§4 acima) |
| tabela de seeds válidas por cenário | `docs/catalogo_respostas.md` (§6 acima) |
| divergências pendentes de curadoria | `docs/reconciliacao_pendente.md` |
| figura | `figures/fig01_distribuicao_status.png` (§5 acima) |

As tabelas em markdown dos dois `.md` foram geradas aqui e **copiadas** para os documentos — não
há escrita automática. Quem reexecutar o notebook e vir número diferente precisa atualizar os
documentos à mão; hoje eles conferem.